In [1]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_experimental.open_clip import OpenCLIPEmbeddings
import glob
import base64

paths = glob.glob('../images/*.jpeg', recursive=True)

In [2]:
lc_docs = []
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

for path in paths:
    doc = Document(
        page_content=encode_image(path),
        metadata ={
            'source': path
        }
    )
    lc_docs.append(doc)

In [3]:
vector_store = FAISS.from_documents(lc_docs, embedding=OpenCLIPEmbeddings())

In [4]:
retriever = vector_store.as_retriever()

In [5]:
import base64
from io import BytesIO

from PIL import Image

def resize_base64_image(base64_string, size=(128, 128)):
    # Decode the Base64 string
    img_data = base64.b64decode(base64_string)
    img = Image.open(BytesIO(img_data))

    # Resize the image
    resized_img = img.resize(size, Image.LANCZOS)

    # Save the resized image to a bytes buffer
    buffered = BytesIO()
    resized_img.save(buffered, format=img.format)

    # Encode the resized image to Base64
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


def is_base64(s):
    try:
        return base64.b64encode(base64.b64decode(s)) == s.encode()
    except Exception:
        return False


def split_image_text_types(docs):
    images = []
    text = []
    for doc in docs:
        doc = doc.page_content  # Extract Document contents
        if is_base64(doc):
            # Resize image to avoid OAI server error
            images.append(
                resize_base64_image(doc)
            )  # base64 encoded str
        else:
            text.append(doc)
    return {"images": images, "texts": text}

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

In [7]:
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def prompt_func(data_dict):
    # Joining the context texts into a single string
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages = []

    # Adding image(s) to the messages if present
    if data_dict["context"]["images"]:
        image_message = {
            "type": "image_url",
            "image_url": {
                "url": f"data:image/jpeg;base64,{data_dict['context']['images'][0]}"
            },
        }
        messages.append(image_message)

    # Adding the text message for analysis
    text_message = {
        "type": "text",
        "text": (
            "As an animal lover, your task is to analyze and interpret images of cute animals, "
            "Please use your extensive knowledge and analytical skills to provide a "
            "summary that includes:\n"
            "- A detailed description of the visual elements in the image.\n"
            f"User-provided keywords: {data_dict['question']}\n\n"
            "Text and / or tables:\n"
            f"{formatted_texts}"
        ),
    }
    messages.append(text_message)

    return [HumanMessage(content=messages)]

# foundation = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)
foundation = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0, 
    max_output_tokens=1024  # In Google it is called 'max_output_tokens'
)

# RAG pipeline
chain = (
    {
        "context": retriever | RunnableLambda(split_image_text_types),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(prompt_func)
    | foundation
    | StrOutputParser()
)

In [8]:
chain.invoke("rottweiler")

'As an animal lover, I\'m delighted to analyze this charming image of a Rottweiler!\n\n**Detailed Description of Visual Elements:**\n\nThe image features a beautiful, healthy **Rottweiler dog** positioned centrally, looking directly at the viewer.\n\n*   **The Dog:**\n    *   **Breed Characteristics:** The dog exhibits the classic traits of a Rottweiler: a robust, muscular build, a broad head, and a short, dense coat.\n    *   **Coloration:** Its fur is predominantly a glossy black, with the characteristic rust or tan markings. These markings are visible on its muzzle, above its eyes (creating expressive "eyebrows"), on its chest, and subtly on its front legs.\n    *   **Facial Features:** The dog\'s eyes are dark, intelligent, and appear alert and friendly. Its nose is large and black. Its mouth is slightly open, revealing a pink tongue that is gently extended, giving the impression of a happy or panting dog. The ears are medium-sized, set high on the head, and appear to be in a natur

In [9]:
docs = retriever.invoke("rottweiler", k=4)

for doc in docs:
    print(doc.metadata)

{'source': '../images/dog_5.jpeg'}
{'source': '../images/dog_3.jpeg'}
{'source': '../images/cat_3.jpeg'}
{'source': '../images/dog_2.jpeg'}
